# <img src="https://intelectacup.com/favicon.ico" /> Intelecta Cup - Data Mining Competition 2025  

---

## Informasi Tim
**Nama Tim:** Cognivio

**Institusi:** Politeknik Negeri Malang

| No | Anggota         | Peran                      | 
|----|----------------------|----------------------------|
| 1  | Vidi Joshubzky Saviola    | Model Developer   |
| 2  | Cakra Wangsa May Ahmad Widodo             | ML Researcher              |
| 3  | Farrel Augusta Dinata             | ML Researcher Lead              |
| 4  | Hidayat Widi Saputra             | Model Developer   |

---

<div style="text-align:center; font-style:italic; opacity:0.8;">
  © 2025 Cognivio Team. All rights reserved.
</div>

# 0. Preparation

In [1]:
import os
import sys
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sklearn
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import make_scorer, mean_absolute_percentage_error
from sklearn.linear_model import Ridge
import xgboost
from xgboost import XGBRegressor
import lightgbm
from lightgbm import LGBMRegressor
from IPython.display import display

import warnings
warnings.filterwarnings('ignore')

# Detect environment
def detect_environment() -> str:
    """Detect if running in Colab, Kaggle, or local environment"""
    if 'google.colab' in sys.modules:
        return 'colab'
    elif 'kaggle_secrets' in sys.modules or os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
        return 'kaggle'
    else:
        return 'local'

ENV = detect_environment()
print(f'🔍 Detected environment: {ENV.upper()}')


print(f'Python version: {sys.version}')
print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')
print(f'Scikit-learn version: {sklearn.__version__}')
print(f'XGBoost version: {xgboost.__version__}')
print(f'LightGBM version: {lightgbm.__version__}')

🔍 Detected environment: LOCAL
Python version: 3.13.0 | packaged by conda-forge | (main, Nov 27 2024, 19:03:49) [MSC v.1942 64 bit (AMD64)]
NumPy version: 2.3.1
Pandas version: 2.2.3
Scikit-learn version: 1.7.2
XGBoost version: 2.1.3
LightGBM version: 4.6.0


# 1. Data Collection

Pada bagian ini merupakan bagian yang memuat dataset yang akan digunakan dalam proyek ini. Notebook ini dirancang agar memudahkan penggunaan di berbagai environment dari lokal, Kaggle, hingga Google Colab. Maka dari itu, proses download dan juga setup dataset dibuat fleksibel menyesuaikan environment notebook pengguna

In [2]:
def setup_dataset_paths(env: str) -> dict[str, str]:
    """Setup dataset paths based on environment"""
    
    # Kaggle environment
    if env == 'kaggle':
        return {
            'train_csv': "/kaggle/input/intelecta-cup-data-mining-competition/train.csv",
            'test_csv': "/kaggle/input/intelecta-cup-data-mining-competition/test.csv",
            'save_dir': "/kaggle/working"
        }
    
    # Local environment
    if env == 'local':
        base_path = "../data" # TODO: Adjust your local environment path
        return {
            'train_csv': f"{base_path}/train.csv",
            'test_csv': f"{base_path}/test.csv",
            'save_dir': f"{base_path}/models"
        }
    
    # Colab environment
    dataset_name = "intelecta-cup-data-mining-competition"
    local_path = "/content/dataset"
    
    # Check if dataset already exists
    if os.path.exists(local_path):
        return {
            'train_csv': f"{local_path}/train.csv",
            'test_csv': f"{local_path}/test.csv",
            'save_dir': "/content/drive/MyDrive/PROJECTS/Cognivio/models"
        }
    
    print("📥 Setting up tabular dataset in Colab...")
    
    # Setup Kaggle credentials
    _setup_kaggle_credentials()
    
    # Download and extract dataset
    _download_and_extract_dataset(dataset_name, local_path)
    
    return {
        'train_csv': f"{local_path}/train.csv",
        'test_csv': f"{local_path}/test.csv",
        'save_dir': "/content/drive/MyDrive/PROJECTS/Cognivio/models"
    }

def _setup_kaggle_credentials() -> None:
    """Setup Kaggle credentials in Colab"""
    kaggle_json_drive = "/content/drive/MyDrive/kaggle.json"
    kaggle_json_local = "/root/.kaggle/kaggle.json"
    
    # Try to copy from Google Drive first
    if os.path.exists(kaggle_json_drive):
        print("📂 Copying kaggle.json from Drive...")
        !mkdir -p /root/.kaggle
        !cp "{kaggle_json_drive}" "{kaggle_json_local}"
        !chmod 600 "{kaggle_json_local}"
        print("✅ Kaggle credentials loaded from Drive")
        return
    
    # Fallback: manual upload
    print("⚠️ Please upload kaggle.json manually or place it in Drive root folder")
    from google.colab import files  # type: ignore
    uploaded = files.upload()
    
    for fn in uploaded.keys():
        !mkdir -p /root/.kaggle
        !mv "{fn}" "/root/.kaggle/kaggle.json"
        !chmod 600 "/root/.kaggle/kaggle.json"
        print(f"✅ Kaggle credentials uploaded: {fn}")


def _download_and_extract_dataset(
    dataset_name: str, 
    local_path: str
) -> None:
    """Download and extract Kaggle competition dataset"""
    import zipfile
    
    # Download dataset
    !kaggle competitions download -c {dataset_name} -p /content
    
    # Extract dataset
    zip_path = f"/content/{dataset_name}.zip"
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Dataset zip not found: {zip_path}")
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(local_path)
    
    print(f"✅ Dataset extracted to {local_path}")

# Setup paths
paths = setup_dataset_paths(ENV)
print(f"📁 Dataset paths configured for {ENV}:")
for key, path in paths.items():
    exists = "✅" if os.path.exists(path) else "⚠️"
    print(f"   {key}: {path} {exists}")

# Create save directory
os.makedirs(paths['save_dir'], exist_ok=True)

# Set random seed for reproducibility
def set_seed(seed: int = 20) -> None:
    random.seed(seed)
    np.random.seed(seed)

📁 Dataset paths configured for local:
   train_csv: ../data/train.csv ✅
   test_csv: ../data/test.csv ✅
   save_dir: ../data/models ✅


# 2. Data Cleaning & Preprocessing

Proses ini bertujuan agar data-data yang tidak sesuai format atau data yang kosong di dataset bisa diolah dengan baik pada model machine learning yang digunakan pada proses selanjutnya. Proses data cleaning dan preprocessing ini melibatkan beberapa teknik berikut:
- Data imputation
- Data encoding
- Transformasi data
- Normalisasi data

In [3]:
df = pd.read_csv(paths['train_csv'])

display(df.head(20))
display(df.describe())
display(df.info())

,ID,Tahun,Nama_Negara,Wilayah,Jenis_Tanaman,Total_Curah_Hujan_mm,Emisi_CO2_JT_Ton,Hasil_Panen_Ton_per_HA,Kejadian_Cuaca_Ekstrim,Akses_Irigasi,Penggunaan_Pestisida_KG_per_HA,Penggunaan_Pupuk_KG_per_HA,Indeks_Kesehatan_Tanah,Strategi_Adaptasi,Suhu_Rata_Rata_C
0,0,2015,USA,South,Soybeans,1658.71,13.36,2.620,10,74.41,38.97,2.64,46.07,Manajemen Air,20.43
1,1,2022,China,East,Wheat,1478.74,9.55,0.570,2,36.90,49.99,77.22,88.87,Rotasi Tanaman,-0.33
2,2,2000,India,'West Bengal',Fruits,1252.34,27.37,2.115,3,34.21,2.75,83.94,77.15,Pertanian Organik,12.97
3,3,2008,Nigeria,'North West',Sugarcane,209.89,16.16,4.158,5,91.74,36.80,37.50,73.59,Pertanian Organik,12.81
4,4,1991,Canada,Ontario,Vegetables,1086.67,3.71,2.430,0,14.72,7.22,28.72,41.90,Tanpa Adaptasi,4.22
5,5,1990,Australia,Queensland,Fruits,2265.29,2.33,2.030,4,79.39,48.80,3.30,77.63,Tanpa Adaptasi,14.70
6,6,2017,Russia,Northwestern,Rice,586.35,7.00,1.070,10,88.96,42.20,71.15,45.82,Tanaman Tahan Kekeringan,7.34
7,7,1995,China,Central,Vegetables,2281.25,24.21,2.907,1,27.44,23.94,13.96,50.97,Tanaman Tahan Kekeringan,16.97
8,8,2016,France,'Provence-Alpes-Cote d’Azur',NaN,792.07,10.31,3.420,1,51.51,4.60,75.99,94.94,Pertanian Organik,30.23
9,9,1991,Nigeria,'North West',Cotton,1746.26,25.28,2.772,7,39.09,9.33,90.96,59.65,Manajemen Air,20.30


,ID,Tahun,Total_Curah_Hujan_mm,Emisi_CO2_JT_Ton,Hasil_Panen_Ton_per_HA,Kejadian_Cuaca_Ekstrim,Akses_Irigasi,Penggunaan_Pestisida_KG_per_HA,Penggunaan_Pupuk_KG_per_HA,Indeks_Kesehatan_Tanah,Suhu_Rata_Rata_C
count,8000.00000,8000.000000,7821.000000,8000.000000,7800.000000,8000.000000,7819.000000,8000.000000,8000.000000,8000.000000,8000.000000
mean,3999.50000,2007.032750,1615.503060,15.271184,2.238219,4.989750,55.394575,24.920015,49.706654,64.824446,15.206680
std,2309.54541,10.106035,807.932322,8.551214,0.996626,3.171814,26.034847,14.454507,28.674985,20.153617,11.490611
min,0.00000,1990.000000,200.170000,0.500000,0.450000,0.000000,10.010000,0.000000,0.030000,30.000000,-4.990000
25%,1999.75000,1998.000000,929.290000,7.860000,1.449000,2.000000,32.905000,12.550000,25.160000,47.115000,5.377500
50%,3999.50000,2007.000000,1614.790000,15.250000,2.170000,5.000000,55.340000,24.930000,49.270000,64.675000,15.140000
75%,5999.25000,2016.000000,2316.820000,22.820000,2.930000,8.000000,77.770000,37.382500,74.430000,82.302500,25.340000
max,7999.00000,2024.000000,2999.670000,30.000000,5.000000,10.000000,99.990000,49.990000,99.990000,100.000000,35.000000


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 15 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   ID                              8000 non-null   int64  
 1   Tahun                           8000 non-null   int64  
 2   Nama_Negara                     8000 non-null   object 
 3   Wilayah                         8000 non-null   object 
 4   Jenis_Tanaman                   7781 non-null   object 
 5   Total_Curah_Hujan_mm            7821 non-null   float64
 6   Emisi_CO2_JT_Ton                8000 non-null   float64
 7   Hasil_Panen_Ton_per_HA          7800 non-null   float64
 8   Kejadian_Cuaca_Ekstrim          8000 non-null   int64  
 9   Akses_Irigasi                   7819 non-null   float64
 10  Penggunaan_Pestisida_KG_per_HA  8000 non-null   float64
 11  Penggunaan_Pupuk_KG_per_HA      8000 non-null   float64
 12  Indeks_Kesehatan_Tanah          80

None

In [4]:
print(f'Total data kosong: {df.isnull().values.sum()}')
display(df.isnull().sum())

Total data kosong: 779


ID                                  0
Tahun                               0
Nama_Negara                         0
Wilayah                             0
Jenis_Tanaman                     219
Total_Curah_Hujan_mm              179
Emisi_CO2_JT_Ton                    0
Hasil_Panen_Ton_per_HA            200
Kejadian_Cuaca_Ekstrim              0
Akses_Irigasi                     181
Penggunaan_Pestisida_KG_per_HA      0
Penggunaan_Pupuk_KG_per_HA          0
Indeks_Kesehatan_Tanah              0
Strategi_Adaptasi                   0
Suhu_Rata_Rata_C                    0
dtype: int64

# 3. Exploratory Data Analysis (EDA)

Pada tahap ini, kami melakukan Exploratory Data Analysis (EDA) yang bertujuan untuk menganalisa dan memvisualisasikan data yang diperoleh serta mampu memahami pola dari dataset.

# 4. Feature Engineering & Selection

Tahapan feature engineering & selection bertujuan untuk mendapatkan fitur-fitur yang sesuai digunakan pada model machine learning nanti. Beberapa fitur baru dibuat guna memperkaya informasi dari dataset. Namun juga beberapa fitur yang kurang relevan dihapuskan karena tidak sesuai dengan tujuan dan model yang digunakan nanti.

# 5. Model Definition

Model definition merupakan tahapan yang kami lakukan untuk memilih model yang tepat dan menyusun model dengan konfigurasi hyperparameter yang sesuai.

# 6. Model Training

Model training merupakan tahapan di mana model berusaha memahami pola dari dataset yang ada sehingga mampu melakukan prediksi dengan tepat.

# 7. Model Evaluation

Ini merupakan tahapan pengujian yang memiliki tujuan untuk mengetahui performa model yang dikembangkan berdasarkan data di luar data training.

# 8. Conclusion